# Interactive Wavelength Solution

[Chapter 9](09-line-identification.ipynb) ended with 72 detected arc lines, each with a sub-pixel
centroid, and six of them identified by hand against the observatory's reference figure. Six
(pixel, wavelength) pairs are not a wavelength solution, but they are the part of one that needs a
person.

The rest follows from an observation: identification and the solution are circular. A good
solution makes matching easy, because it predicts each line's position to within a fraction of a
pixel and the ambiguity disappears. A good set of matches makes the solution easy, because fitting
a polynomial to known pairs is trivial. Neither comes first, so you bootstrap from the anchors.

In `specreduce.wavecal1d.WavelengthCalibration1D` that is two calls. `fit_lines` fits a
polynomial through the anchors and uses it to match every detection to the catalog line predicted
nearest to it. `refine_fit` refits the polynomial through all the matched pairs and matches again
with a tighter tolerance, now that the solution predicts positions to a fraction of a pixel.
With anchors chosen well, strong, isolated, and spread over the detector, one call of each is the
whole solution. Further rounds of `refine_fit` with a tighter tolerance are available for anchors that
are few or poorly placed, but they should not be needed.

This chapter runs those steps one at a time, stopping after each to look at what happened.
[Chapter 11](11-wavelength-solution-automated.ipynb) does the same job without the human, starting
from prior knowledge of the solution instead of from anchors.

In [ ]:
import sys
import warnings

sys.path.append("../src")

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from astropy.modeling import models
from specreduce.background import Background
from specreduce.extract import BoxcarExtract
from specreduce.tracing import FitTrace, FlatTrace
from specreduce.tilt_solution import TiltSolution
from specreduce.wavecal1d import WavelengthCalibration1D
from specreduce.wavesol1d import WavelengthSolution1D

from common import (
    ANCHOR_PIXELS,
    ANCHOR_WAVELENGTHS,
    GTC_HGAR,
    TILT_SOLUTION_FILE,
    WAVELENGTH_SOLUTION_FILE,
    fit_tilt_solution,
    read_data,
)

plt.rc("figure", figsize=(8, 2.4), dpi=110)
plt.rc("font", size=9)

arcs, lamps, obj = read_data()
nx = obj.data.shape[1]

## Where Chapter 9 left off

The inputs are the same: 1D arc spectra extracted at the reference row of the rectified frames,
and the six anchors, imported from `common.py`. The tilt solution comes from the ASDF file
Chapter 6 wrote, and the extraction is repeated in a few lines so the chapter runs on its own. The
line detection is not repeated: this time the calibration class runs it.

In [ ]:
reference = FlatTrace(arcs[0], 135)

if not TILT_SOLUTION_FILE.exists():  # normally written by Chapter 6
    fit_tilt_solution(arcs).to_asdf(TILT_SOLUTION_FILE)

solution = TiltSolution.from_asdf(TILT_SOLUTION_FILE)

rectified_arcs = [solution.resample(frame) for frame in arcs]
rectified_obj = solution.resample(obj)


spectra = [
    BoxcarExtract(frame, reference, width=15).spectrum for frame in rectified_arcs
]

## Setting up the calibration

`WavelengthCalibration1D` holds the observations, the catalog lines and the fitted model, and
provides the methods that move between them. The observations go in one of two ways, on an equal
footing. `arc_spectra` takes the extracted 1D spectra, and the class then finds the lines itself
with its `find_lines` method. `obs_lines` takes pixel positions of lines that have already been
measured, optionally paired with amplitudes, one array per arc. The first is what most reductions
do and what this chapter does. The second is for line positions measured with a detection routine
of your own rather than with `find_arc_lines`; it needs `pix_bounds` and `ref_pixel` given
explicitly, since there is no spectrum to read them from, and it forgoes the plots that draw the
spectra.

The `line_lists` argument takes one entry per arc spectrum, and each entry is either a list of ion
names to pull from `pypeit`, as Chapter 9 did, or a plain array of wavelengths. We start with a
single lamp and an array: `GTC_HGAR`, the observatory's list for HgAr through R1000R that
Chapter 9 introduced.

In [ ]:
calibration = WavelengthCalibration1D(
    arc_spectra=[spectra[0]],
    line_lists=[GTC_HGAR],
    ref_pixel=512,
    unit=u.angstrom,
)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # unconverged refits on non-line detections
    calibration.find_lines(fwhm=2.5, noise_factor=15)

print(f"observed lines: {[len(lines) for lines in calibration.observed_lines]}")
print(f"catalog lines: {[len(lines) for lines in calibration.catalog_lines]}")

An instrument-specific list is worth using when one exists. It holds the lines this spectrograph
actually shows at this resolution, eleven of them, rather than every transition the species can
emit, and Chapter 9 showed that all but the blended Hg pair near 5780 Å have a detection to match.

`find_lines` runs `find_arc_lines` on each arc spectrum, with the `fwhm` and `noise_factor`
Chapter 9 explained, and stores the result in `observed_lines`. It applies no width cut, so the
detections Chapter 9 rejected are still in the list; the matching tolerance used below keeps
them out of the fit.

:::{note}
The detections live in `observed_lines`, and the attribute can be assigned directly, in the same
array format `obs_lines` takes. That is how positions from another detection routine go in
when the spectra are still wanted for plotting.
:::

Before fitting anything, look at the two lists side by side. The top panel is the catalog in
wavelength space, the bottom the detections in pixel space. With no solution yet there is nothing
relating them.

In [ ]:
fig = calibration.plot_fit(figsize=(8, 4.2), label_kwargs=dict(size=6))

# Widen the wavelength axis so the outermost catalog lines do not sit on the spines.
fig.axes[0].set_xlim(GTC_HGAR.min() - 150, GTC_HGAR.max() + 150);

## Fitting the anchors

The six anchors from Chapter 9 are the pairs that are certainly right. `fit_lines` takes them and
fits the polynomial. Two conveniences matter: `match_obs` and `match_cat` **snap** the values you
supply to the nearest entries in the observed and catalog lists, so the numbers can be
approximate. The anchor pixels were read off a plot to the nearest whole number, and the class
substitutes the measured centroids. Six lines are enough for a cubic, and `degree=3` is what the
rest of the chapter uses:

In [ ]:
solution = calibration.fit_lines(
    pixels=ANCHOR_PIXELS,
    wavelengths=ANCHOR_WAVELENGTHS,
    degree=3,
    match_obs=True,
    match_cat=True,
    refine_fit=True,
)

print(
    f"rms: {calibration.rms('wavelength'):.3f} A  ({calibration.rms('pixel'):.3f} pix)"
)
print(f"coverage: {solution.pix_to_wav(0):.0f} to {solution.pix_to_wav(nx - 1):.0f} A")
print(
    f"dispersion at center: "
    f"{solution.pix_to_wav(513) - solution.pix_to_wav(512):.3f} A/pix"
)

That is already a good solution, and two independent checks say so.

The **coverage** comes out at 5097–10257 Å against the grism's documented 5100–10000 Å, and
nothing in the fit knew that figure. The blue end agrees to three Ångström, from six lines and a
cubic. The red end runs about 260 Å past the nominal value, which is unsurprising: the quoted
range is where the grism is *useful*, and the fit simply extrapolates its cubic to the last pixel,
well beyond the reddest anchor at 9123 Å. Treat the red extreme as extrapolation, not measurement.

The **rms** is 0.48 Å, about 0.089 pixels.

`fit_lines` did more than fit six points. With `refine_fit=True`, its default, it took the anchor
solution, matched every detection to the catalog line predicted within `refine_max_distance` of
it, 5 Å by default, and refitted with everything it found.

In [ ]:
calibration.plot_residuals(space="wavelength");

## Refining

`refine_fit` is the second call. It refits the polynomial through all the matched pairs and
matches again with the tolerance given as `max_match_distance`, repeating internally until the
matched set stops changing. The 5 Å inside `fit_lines` was generous, chosen to catch every line
from a solution fitted to six points. With the rms now under half an Ångström, 2 Å is the safer
tolerance: a few times the rms, and well under the spacing of neighboring catalog lines.

In [ ]:
calibration.refine_fit(max_match_distance=2.0)
calibration.remove_unmatched_lines()

print(f"lines retained: {[len(lines) for lines in calibration.observed_lines]}")
print(
    f"final rms: {calibration.rms('wavelength'):.3f} A "
    f"({calibration.rms('pixel'):.3f} pix)"
)

fig = calibration.plot_fit(figsize=(8, 4.2), obs_to_wav=True, label_kwargs=dict(size=6))
for ax in fig.axes:
    ax.set_xlim(GTC_HGAR.min() - 150, GTC_HGAR.max() + 150)

Nothing changed: the same nine lines, the same 0.48 Å. That is the normal outcome with good
anchors. The matching pass inside `fit_lines` had already found every line the catalog offers,
and the refinement confirms the solution rather than moving it.

`remove_unmatched_lines` discards the detections that never found a partner, leaving only the
identified set. With `obs_to_wav=True` the observed lines are now drawn in wavelength space
through the fitted solution, so the two panels are directly comparable and the detections land on
their catalog counterparts.

Nine of the eleven GTC lines are used. The two that are missing, 5769.598 and 5790.663 Å, form a
close pair predicted around pixels 157 and 161, where the finder returns no detection at all.
Blended at this resolution, they form one broad feature that no single Gaussian fits.

## Using more than one lamp

Everything so far used HgAr alone, and it covers the range unevenly: nine lines, none of them
between pixels 87 and 405. The `line_lists` and `arc_spectra` arguments both accept sequences, so
adding the other two lamps is a change of arguments rather than of workflow: the same `fit_lines`
with the same six HgAr anchors, then the same `refine_fit` at 2 Å. The solution is one polynomial
shared by all three spectra, so anchors identified on one lamp predict where every catalog line
of the others falls, and no anchors are needed for Ne or Xe. For those two there is no GTC list,
so `pypeit` ion names are used, with `wave_air=True` because `pypeit` wavelengths are in vacuum
while the GTC list is in air.

In [ ]:
multi = WavelengthCalibration1D(
    arc_spectra=spectra,
    line_lists=[GTC_HGAR, ["NeI"], ["XeI"]],
    line_list_bounds=(5000, 10500),
    ref_pixel=512,
    unit=u.angstrom,
    wave_air=True,
)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # unconverged refits on non-line detections
    multi.find_lines(fwhm=2.5, noise_factor=15)

multi.fit_lines(
    pixels=ANCHOR_PIXELS,
    wavelengths=ANCHOR_WAVELENGTHS,
    degree=3,
    match_obs=True,
    match_cat=True,
    refine_fit=True,
)
multi.refine_fit(max_match_distance=2.0)

# Record which detections the refinement matched, and their amplitudes, before the
# unmatched ones are dropped; the amplitude comparison below needs both groups.
matched_masks = [
    ~np.ma.getmaskarray(np.ma.asarray(lines))[:, 0] for lines in multi.observed_lines
]
amplitudes = [np.asarray(lines.data)[:, 1] for lines in multi.observed_lines]
matched_per_lamp = [int(mask.sum()) for mask in matched_masks]
total_detected = sum(len(lines) for lines in multi.observed_lines)
catalog_per_lamp = [len(lines) for lines in multi.catalog_lines]
multi.remove_unmatched_lines()

print(f"catalog lines: {dict(zip(lamps, catalog_per_lamp))}")
print(f"matched:         {dict(zip(lamps, matched_per_lamp))}")
print(f"total matched:   {sum(matched_per_lamp)} of {total_detected}")
print(f"rms: {multi.rms('wavelength'):.3f} A ({multi.rms('pixel'):.3f} pix)")
print(
    f"coverage: {multi.solution.pix_to_wav(0):.0f} to "
    f"{multi.solution.pix_to_wav(nx - 1):.0f} A"
)

Two calls again, and 28 matched lines across the three lamps: the HgAr anchor solution placed
the Ne and Xe catalog lines closely enough for the matching inside `fit_lines` to pick them up,
and the refinement settled the set. The three-lamp solution has a **higher** rms than the
single-lamp one, 0.66 Å against 0.48, and is nevertheless the better solution. The rms went up because the fit now has to satisfy 28 lines
spread across the whole detector instead of 9 clustered in its red half, including fainter lines
whose centroids are less precise. The single-lamp number was flattering: a cubic through nine
points in half the range is barely constrained in the other half, and its rms says nothing about
how it behaves there.

**Do not choose a solution by its rms alone.** Coverage and line count matter at least as much,
and the two solutions agree on the endpoints to within a couple of Ångström, which is the real
evidence that both are sound.

Only 28 of 89 detections were matched. Chapter 9 anticipated part of that: many catalog lines
are too faint to appear, some detections have no catalog counterpart, and without a width cut the
list also holds the spurious detections Chapter 9 showed how to recognize. The amplitudes of the
two groups tell most of the story:

In [ ]:
print(
    f"{'lamp':>5}  {'matched':>8}  {'median amp':>11}  {'unmatched':>10}  {'median amp':>11}"
)
for lamp, amplitude, matched in zip(lamps, amplitudes, matched_masks):
    print(
        f"{lamp:>5}  {matched.sum():>8}  {np.median(amplitude[matched]):>11.0f}  "
        f"{(~matched).sum():>10}  {np.median(amplitude[~matched]):>11.0f}"
    )

On HgAr and Xe the unmatched detections are an order of magnitude fainter than the matched ones:
weak features, blends, and failed refits. Ne is the exception, because its unresolved combs of
bright lines yield detections with no single catalog counterpart. Leaving all of them out is
correct, since a poor centroid drags the fit.

The identified lines and the residuals, for all three lamps:

In [ ]:
fig = multi.plot_fit(figsize=(8, 7.5), obs_to_wav=True, label_kwargs=dict(size=6))
for ax in fig.axes:
    ax.set_xlim(5000, 10500)

multi.plot_residuals(space="wavelength");

The three lamps divide the range between them: HgAr and Ne share the blue and middle, Xe alone
holds the far red. The residuals scatter to about 2 Å at most with no trend against wavelength,
which is what a polynomial of the right degree through correctly identified lines looks like.

## Saving the solution

Like the tilt solution of Chapter 6, a wavelength solution is a calibration product: it belongs to
the instrument configuration, not to any one exposure, and every frame taken in that configuration
shares it. `WavelengthSolution1D` serializes to ASDF the same way, storing the fitted polynomial as
an analytic model.

The ASDF file is the book's own record of the solution, but it is not the only form the solution
takes. For code that has its own way of handling a wavelength axis, `WavelengthSolution1D` speaks
the two standard languages. The `gwcs` property exposes the exact pixel-to-wavelength model as a
GWCS object, and the `wcs` method fits the FITS WCS grism dispersion function of
[Greisen et al. (2006)](https://ui.adsabs.harvard.edu/abs/2006A%26A...446..747G/abstract) to it
and returns an `astropy.wcs.WCS`, with the spectral axis typed `AWAV-GRA` for air wavelengths or
`WAVE-GRI` for vacuum. Here the FITS approximation stays within 0.2 Å of the exact cubic, four
hundredths of a pixel, across the whole detector.

`from_asdf` reads it back; Chapter 12 does so and chains it with the tilt solution.

In [ ]:
multi.solution.to_asdf(WAVELENGTH_SOLUTION_FILE)
restored = WavelengthSolution1D.from_asdf(WAVELENGTH_SOLUTION_FILE)

## Applying the solution

A `WavelengthSolution1D` converts between pixel and wavelength, and there are two ways to put that
on a spectrum. `resample` rebins a pixel-space spectrum onto a uniform wavelength grid, conserving
the integrated flux and returning a flux **density**. `attach_wcs` leaves the samples on the pixel
grid and attaches the FITS WCS instead. Here `resample` is applied to the science spectrum from
Chapter 6, and `attach_wcs` follows.

In [ ]:
trace = FitTrace(
    rectified_obj, bins=32, peak_method="gaussian", trace_model=models.Chebyshev1D(4)
)
subtracted = Background.two_sided(
    rectified_obj, trace, separation=60, width=20
).sub_image()
science = BoxcarExtract(subtracted, trace, width=11).spectrum

calibrated = multi.solution.resample(science)

print(
    f"spectral axis: {calibrated.spectral_axis[0]:.1f} to {calibrated.spectral_axis[-1]:.1f}"
)
print(f"flux unit:     {calibrated.flux.unit}")

# resample returns a flux *density*, so the conserved quantity is the integral, not the
# sum. resample records the bin edges it actually used, so this integral is exact.
edges = calibrated.spectral_axis.bin_edges.to(u.AA).value
integrated = np.nansum(calibrated.flux.value * np.diff(edges))
print(f"flux retained: {integrated / np.nansum(science.flux.value):.4f}")
print(f"uncertainty:   {type(calibrated.uncertainty).__name__}")

In [ ]:
fig, axs = plt.subplots(2, figsize=(8, 4.2), constrained_layout=True)

axs[0].plot(science.spectral_axis.value, science.flux.value, lw=0.6)
axs[0].set(xlim=(0, nx - 1), xlabel="Dispersion [pix]", ylabel="Flux [e$^-$]")

axs[1].plot(calibrated.spectral_axis.value, calibrated.flux.value, lw=0.6, c="C1")
axs[1].set(
    xlim=(calibrated.spectral_axis.value[0], calibrated.spectral_axis.value[-1]),
    xlabel=r"Wavelength [$\AA$]",
    ylabel=f"Flux [{calibrated.flux.unit.to_string('latex_inline')}]",
);

The horizontal axis is now physical. The absorption features that Chapter 1 could only point at
sit at wavelengths that can be looked up: the deep bands in the red are telluric O₂ and H₂O, and
the stellar lines can be identified against a template.

Two things about that output are worth stating plainly.

**The vertical axis has changed quantity.** `resample` returns a flux **density**, in `electron /
Angstrom`, not counts per bin, which is why the flux retained above is computed as an integral
rather than a ratio of sums. The practical rule is that a density is what you want for anything
that will be flux-calibrated or compared between instruments, and that you must integrate, not
sum, to get back to electrons.

**Flux is conserved exactly**, to the four decimal places printed. Note how that number is
obtained: by multiplying the density by the width of each bin and summing. The default grid is
linear, here 1000 bins of 5.17 Å spanning the solution's range, so every bin has the same width,
but `resample` accepts any bin edges, and the habit worth forming is to take the widths from
`spectral_axis.bin_edges`, which records the edges actually used. Differencing the bin centers
instead yields one width too few and reports a spurious loss of about one part in ten thousand.

The uncertainty is propagated through the rebinning, as it is through the tilt correction's
`resample` in Chapter 6.

### Keeping the pixel grid

Resampling is not always wanted. It correlates neighboring bins, and a spectrum bound for a
fitting code that models the instrument itself is better left on its native pixels. `attach_wcs`
returns a copy of the pixel-space spectrum with the fitted FITS WCS as its spectral axis, the
samples untouched. That is the form to serialize: written with the `specutils` `wcs1d-fits`
format, the flux is stored as an image and the wavelength solution as header keywords that any
FITS reader understands.

In [ ]:
with_wcs = multi.solution.attach_wcs(science)

## Summary

- Identification and the wavelength solution are circular, so bootstrap from a few certain
  anchors: **`fit_lines` fits them and matches the rest, `refine_fit` refits and matches again
  with a tighter tolerance.** With good anchors one call of each is the solution.
- **`WavelengthCalibration1D` takes either arc spectra or measured line positions.** Pass
  `arc_spectra` and let `find_lines` detect the lines, as most reductions do, or pass `obs_lines`
  with positions measured by a detection routine of your own.
- Prefer an **instrument-specific line list** when one exists. GTC's eleven-line OSIRIS list
  holds the lines this setup shows, so nearly every catalog entry has a detection to match.
- `fit_lines(..., match_obs=True, match_cat=True)` snaps approximate inputs to the measured
  centroids and catalog values, so anchors read off a plot are good enough. Six were enough for
  a cubic, and the call already matches the remaining lines within 5 Å.
- **Check the coverage against the instrument documentation.** The fit knew nothing of the
  grism's nominal range and reproduced its blue end to three Ångström; the red end is
  extrapolation beyond the last line.
- `refine_fit` at a tighter tolerance, 2 Å here, confirms the solution rather than moving it.
  Further rounds are a fallback for weak anchors, and tightening too far discards correctly
  identified lines while the rms improves for the wrong reason.
- Adding lamps is a change of arguments, not of workflow: `arc_spectra` and `line_lists` take
  sequences, anchors identified on one lamp serve all of them, and `wave_air=True` is needed with
  `pypeit` lists, whose wavelengths are in vacuum.
- **Do not choose a solution by its rms alone.** One lamp gave 0.48 Å from 9 lines in half the
  range; three lamps gave 0.66 Å from 28 lines over the whole detector, and the second is the
  better solution. Judge coverage and line count as well.
- **Serialize the solution with `to_asdf`.** It is a calibration product shared by every frame
  in the configuration; `from_asdf` reads it back, and Chapter 12 chains it with the tilt
  solution.
- The solution is also available as a **GWCS** (`gwcs`) and a **FITS WCS** (`wcs`) for other
  code, and `attach_wcs` puts the FITS WCS on a pixel-space spectrum so it can be written without
  resampling.
- **`resample` returns a flux density**, in `electron / Angstrom`, on a linear wavelength grid,
  with the uncertainty propagated. Recover a total by integrating against
  `spectral_axis.bin_edges`, never by summing the values.

Next: [Automated Wavelength Solution](11-wavelength-solution-automated.ipynb), which starts from
prior knowledge of the solution instead of from anchors.